In [1]:
import geopandas as gpd
import pandas as pd
import requests
import time

In [2]:
divisions_geojson_path = "../raw_data/LAPD_Division.geojson"

# Load divisions geojson
divisions = gpd.read_file(divisions_geojson_path)

# Keep only needed columns
divisions = divisions[["OBJECTID", "APREC", "geometry"]]

# Convert to WGS84 (lat/lon)
divisions = divisions.to_crs("EPSG:4326")

# Compute centroid
divisions["centroid"] = divisions.geometry.centroid

# Extract latitude and longitude
divisions["lat"] = divisions.centroid.y
divisions["lon"] = divisions.centroid.x

# Keep only what we need
divisions = divisions[["APREC", "lat", "lon"]]
divisions.head()

C:\Users\Milena\AppData\Local\Temp\ipykernel_20004\2270359206.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  divisions["centroid"] = divisions.geometry.centroid
C:\Users\Milena\AppData\Local\Temp\ipykernel_20004\2270359206.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  divisions["lat"] = divisions.centroid.y
C:\Users\Milena\AppData\Local\Temp\ipykernel_20004\2270359206.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  divisions["lon"] = divisions.centroid.x


,APREC,lat,lon
0,MISSION,34.277186,-118.448127
1,DEVONSHIRE,34.265559,-118.539474
2,FOOTHILL,34.252402,-118.345875
3,TOPANGA,34.190021,-118.610929
4,WEST VALLEY,34.174159,-118.517739


In [3]:
divisions["APREC"].unique()

array(['MISSION', 'DEVONSHIRE', 'FOOTHILL', 'TOPANGA', 'WEST VALLEY',
       'NORTH HOLLYWOOD', 'VAN NUYS', 'NORTHEAST', 'HOLLYWOOD',
       'WEST LOS ANGELES', 'HOLLENBECK', 'RAMPART', 'WILSHIRE', 'OLYMPIC',
       'SOUTHWEST', 'NEWTON', 'PACIFIC', '77TH STREET', 'SOUTHEAST',
       'HARBOR', 'CENTRAL'], dtype=object)

In [4]:
START_YEAR = 2010
END_YEAR = 2019
all_weather_data = []

In [5]:
for index, row in divisions.iterrows():

    division_name = row["APREC"]
    lat = row["lat"]
    lon = row["lon"]

    for year in range(START_YEAR, END_YEAR + 1):

        print(f"Downloading {division_name} - {year}")

        url = "https://archive-api.open-meteo.com/v1/archive"

        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": f"{year}-01-01",
            "end_date": f"{year}-12-31",
            "hourly": "temperature_2m,precipitation,wind_speed_10m,visibility,rain,showers,snowfall",
            "timezone": "America/Los_Angeles"
        }

        success = False

        while not success:

            response = requests.get(url, params=params)
            data = response.json()

            # RATE LIMIT DETECTED
            if "reason" in data and "limit exceeded" in data["reason"].lower():
                print("Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
                continue

            # OTHER ERROR
            if "hourly" not in data:
                print("Unexpected error:")
                print(json.dumps(data, indent=4))
                break

            # SUCCESS
            hourly = data["hourly"]

            weather_df = pd.DataFrame({
                "datetime": hourly["time"],
                "temperature": hourly["temperature_2m"],
                "precipitation": hourly["precipitation"],
                "wind_speed": hourly["wind_speed_10m"],
                "visibility": hourly["visibility"],
                "rain": hourly["rain"],
                "showers": hourly["showers"],
                "snowfall": hourly["snowfall"],
            })

            weather_df["APREC"] = division_name
            weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

            all_weather_data.append(weather_df)

            success = True

            time.sleep(2)  # small delay between successful calls

In [6]:
# Combine everything
weather_all = pd.concat(all_weather_data, ignore_index=True)

In [11]:
weather_all.head()

,datetime,temperature,precipitation,wind_speed,visibility,rain,showers,snowfall,APREC
0,2010-01-01 00:00:00,6.2,0.0,9.3,None,0.0,0.0,0.0,MISSION
1,2010-01-01 01:00:00,5.8,0.0,9.1,None,0.0,0.0,0.0,MISSION
2,2010-01-01 02:00:00,5.6,0.0,9.8,None,0.0,0.0,0.0,MISSION
3,2010-01-01 03:00:00,5.4,0.0,10.2,None,0.0,0.0,0.0,MISSION
4,2010-01-01 04:00:00,5.4,0.0,10.7,None,0.0,0.0,0.0,MISSION


In [12]:
weather_all.shape

(1840608, 9)

In [9]:
weather_all.isnull().sum()

datetime               0
temperature            0
precipitation          0
wind_speed             0
visibility       1840608
rain                   0
showers                0
snowfall               0
APREC                  0
dtype: int64

In [19]:
# Drop visibility column since there is no data there
weather_all.drop(columns=["visibility"], inplace=True)

In [20]:
# Number of records with precipitaion value != 0
(weather_all['precipitation'] != 0).sum()

83475

In [21]:
# Number of records with rain value != 0
(weather_all['rain'] != 0).sum()

83464

In [22]:
# Number of records with rain value != 0
(weather_all['showers'] != 0).sum()

0

In [23]:
# Drop showers column since there is no data there
weather_all.drop(columns=["showers"], inplace=True)

In [24]:
# Number of records with rain value != 0
(weather_all['snowfall'] != 0).sum()

63

In [ ]:
# Drop snowfall column since there is no usefull data there
weather_all.drop(columns=["snowfall"], inplace=True)

In [25]:
# Export the processed DataFrame to CSV
weather_all.to_csv("../processed_data/weather_2010_2019_hourly.csv", index=False)